# Reference effects

Reproduce `reference_effects_new.pdf`: the ridge-optimized risk landscape in
common differing-site mean $u=m_{\neq}=\tilde m_{\neq}$ and aspect ratio $r$,
followed by exact cuts at fixed $r$ and fixed $u$.

Run all cells to load the supplied numerical results and save the PDF in `output/`.
Set `RECOMPUTE=True` to evaluate changed scientific settings; this can be slow.
Style controls and all figure construction remain editable below.

The comparison holds $m=0$, $F=\tilde F$, $F_{\neq}=\tilde F_{\neq}$,
and $f=\tilde f$, so the reference response means and $R_\infty$ remain constant.
The plotted risk omits the independent test-noise floor and is normalized by
$R_\infty$. Dashed map guides identify the cuts; dotted unit lines mark the
constant-predictor risk.

In [ ]:
from pathlib import Path
from dataclasses import asdict
from copy import deepcopy
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'dms').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'dms').is_dir():
    raise RuntimeError('Start in the reproducibility root or its notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
for name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory',
             'dms.jobs', 'dms.optimization', 'dms.ridge_diagnostics', 'dms.exploration',
             'dms.result_cache', 'dms.reference_compensation',
             'dms.reference_compensation_grid'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
    else:
        importlib.import_module(name)
from dms.exploration import SweepParameters
from dms.result_cache import ResultCache
from dms.reference_compensation_grid import cached_compensation_grid
from tools.portable_results import load_results, save_results, assert_settings_match

plt.rcParams.update({'axes.labelsize': 11, 'axes.labelweight': 'bold',
                     'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11})
OUTPUT_DIR = PROJECT_ROOT / 'output'
DATA_ARCHIVE = PROJECT_ROOT / 'data' / 'reference_effects'
FIGURES = {}


## Scientific controls

In [ ]:
# False: load the supplied numerical results. True: evaluate the selected settings.
RECOMPUTE = False
FORCE_RECOMPUTE = False
BASE = SweepParameters(q=20, sigma=1., G2=100., f=.2, ftilde=.2,
                       eta=.2, F=.4, FTest=.4, Fneq=.2, rhoneq=0.,
                       m=0., mneq=0., mneqtilde=0.)
MEAN_CONVENTION = 'normalized'
REFERENCE_L = 100
SEARCH = dict(method='nonnegative', lambda_max=1000., n_grid=641)

# Landscape and exact cuts have independent grids.
MAP_MEANS = np.linspace(0., .65, 51)
MAP_R = np.geomspace(.1, 10., 51)
CUT_MEANS = np.linspace(0., .65, 81)
FIXED_R = [.1, .3, 1., 3., 10.]
FIXED_MEANS = [0., .2, .4, .6]
CUT_R = np.geomspace(.1, 10., 81)

CACHE = dict(directory=PROJECT_ROOT / '.cache' / 'reference_effects',
             enabled=True, verbose=True)


## Load or compute each stage

In [ ]:
def reference_settings():
    return dict(base=asdict(BASE), mean_convention=MEAN_CONVENTION,
                reference_L=REFERENCE_L, search=deepcopy(SEARCH),
                stages=dict(map=dict(mean_values=MAP_MEANS, r_values=MAP_R),
                            mean_cuts=dict(mean_values=CUT_MEANS, r_values=FIXED_R),
                            r_cuts=dict(mean_values=FIXED_MEANS, r_values=CUT_R)))


def reference_stage(stage):
    settings = reference_settings()
    if not RECOMPUTE:
        stored, metadata = load_results(DATA_ARCHIVE, restore_records=True)
        assert_settings_match(metadata['settings'], settings)
        data = stored[stage]
        identity = metadata['source_identities'][stage]
    else:
        data, identity = cached_compensation_grid(
            BASE, **settings['stages'][stage], stage=stage,
            mean_convention=MEAN_CONVENTION, reference_L=REFERENCE_L,
            search=SEARCH, cache=ResultCache(**CACHE),
            compute=True, force=FORCE_RECOMPUTE)
    if not np.all(np.isin(data['status'], ['zero', 'interior'])):
        raise RuntimeError(f'{stage}: unresolved or infeasible results; inspect data before plotting.')
    print(stage, dict(zip(*np.unique(data['status'], return_counts=True))))
    return data, identity


In [ ]:
MAP_DATA, MAP_ID = reference_stage('map')

In [ ]:
MEAN_DATA, MEAN_ID = reference_stage('mean_cuts')

In [ ]:
R_DATA, R_ID = reference_stage('r_cuts')

## Editable figure construction

In [ ]:
def refeq_risk(data, normalization):
    values = np.asarray(data['risk'], dtype=float)
    key = {'Rinf': 'r_infinity', 'sigma2': 'sigma2', 'none': None}[normalization]
    if key is not None:
        divisor = np.asarray(data[key], dtype=float)
        with np.errstate(divide='ignore', invalid='ignore'):
            values = np.where(divisor > 0, values/divisor, np.nan)
    return np.where(np.isin(data['status'], ['zero', 'interior', 'upper']), values, np.nan)


def refeq_colours(count, panel):
    colours = panel['colours']
    if colours is None:
        colours = plt.get_cmap(panel['cmap'])(np.linspace(*panel['colour_range'], count))
    if len(colours) < count:
        raise ValueError('Provide one colour for every requested curve, including invalid curves')
    return colours


def refeq_limits(arrays, explicit=None, *, padding=.04):
    if explicit is None:
        flat = np.concatenate([np.asarray(v, dtype=float).ravel() for v in arrays])
        finite = flat[np.isfinite(flat)]
        lo, hi = (float(finite.min()), float(finite.max())) if finite.size else (0., 1.)
        pad = padding*(hi-lo) if hi > lo else .05*max(1., abs(lo))
        explicit = (lo-pad, hi+pad)
    values = np.asarray(explicit, dtype=float)
    if values.shape != (2,) or np.any(~np.isfinite(values)) or values[0] >= values[1]:
        raise ValueError('Limits must contain two finite increasing values')
    return tuple(values)


def refeq_series(data, direction, normalization):
    risk = refeq_risk(data, normalization)
    if direction == 'mean':
        return [(float(r), data['mean_values'], risk[i], data['feasible'][i])
                for i, r in enumerate(data['r_values'])]
    return [(float(u), data['r_values'], risk[:, i], data['feasible'][:, i])
            for i, u in enumerate(data['mean_values'])]


def draw_refeq_cut(ax, series, options, panel):
    reference = dict(panel['reference_line'])
    if reference.pop('show') and options['normalization'] == 'Rinf':
        ax.axhline(reference.pop('value'), **reference)
    for (value, x, y, feasible), colour in zip(series, refeq_colours(len(series), panel)):
        style = dict(color=colour, linewidth=panel['linewidth'], linestyle=panel['linestyle'],
                     label=panel['label_format'].format(value=value))
        style.update(panel['curve_overrides'].get(value, {}))
        finite = np.isfinite(y)
        if not np.any(finite):
            style['label'] += ' (infeasible)' if not np.any(feasible) else ' (no valid result)'
            ax.plot([], [], **dict(style, linestyle='none', marker='x',
                                  markersize=panel['isolated_marker_size']))
        else:
            ax.plot(x, y, **style)  # preserve NaN gaps
            if panel['show_isolated_points']:
                isolated = finite & ~np.r_[False, finite[:-1]] & ~np.r_[finite[1:], False]
                ax.plot(x[isolated], y[isolated], linestyle='none', color=style['color'],
                        marker=panel['isolated_marker'], markersize=panel['isolated_marker_size'],
                        clip_on=False)
    ax.set_xscale(panel['xscale'])
    ax.set_xlabel(panel['xlabel'], fontsize=options['label_size'], fontweight='bold')
    if panel['xlim'] is not None: ax.set_xlim(panel['xlim'])
    if panel['xticks'] is not None: ax.set_xticks(panel['xticks'])
    ax.set_title(panel['title'], fontsize=options['title_size'])
    if panel['show_legend']:
        ax.legend(fontsize=options['legend_size'], **panel['legend'])


def draw_refeq_risk_strip(ax, limits, norm, options, map_style):
    """Library-breadth colourbar style, on the shared risk coordinates."""
    from matplotlib.patches import Polygon
    bar = options['colorbar']
    count = options['bar_resolution']
    if int(count) != count or count < 2:
        raise ValueError('bar_resolution must be an integer >= 2')
    cmap = plt.get_cmap(map_style['cmap'])
    edges = np.linspace(*limits, int(count)+1)
    centers = (edges[:-1]+edges[1:])/2
    ax.pcolormesh([0., 1.], edges, centers[:, None], cmap=cmap, norm=norm,
                  shading='flat', rasterized=True)
    ax.set_xlim(0., 1.)
    ax.set_ylim(limits)
    ax.set_xticks([])
    ax.tick_params(axis='y', which='both', left=False, right=True,
                   labelleft=False, labelright=True,
                   pad=bar['tickpad'], length=bar['tick_length'])
    for spine in ax.spines.values():
        spine.set_color(bar['edgecolor'])
        spine.set_linewidth(bar['linewidth'])
    extension = bar['extend']
    if extension not in ('neither', 'min', 'max', 'both'):
        raise ValueError("Colourbar extend must be 'neither', 'min', 'max' or 'both'")
    size = bar['extension_size']
    for end, vertices, value in (
        ('min', [(0., 0.), (1., 0.), (.5, -size)], limits[0]),
        ('max', [(0., 1.), (1., 1.), (.5, 1.+size)], limits[1]),
    ):
        if extension in (end, 'both'):
            ax.add_patch(Polygon(vertices, transform=ax.transAxes,
                facecolor=cmap(norm(value)), edgecolor=bar['edgecolor'],
                linewidth=bar['linewidth'], clip_on=False))


def draw_reference_equal_r(data, mean_data, r_data, options, map_style, panels):
    """All figure construction is editable here; inputs are stored numerical arrays."""
    from matplotlib.colors import Normalize, ListedColormap
    from matplotlib.patches import Patch
    from matplotlib.ticker import FixedLocator, NullLocator
    import matplotlib.patheffects as path_effects
    if min(len(data['mean_values']), len(data['r_values'])) < 2:
        raise ValueError('The map needs at least two points on each axis')
    if any(other['metadata']['base'] != data['metadata']['base'] for other in (mean_data, r_data)):
        raise ValueError('All panels must use the same controlled baseline')
    risk = refeq_risk(data, options['normalization'])
    series = [refeq_series(mean_data, 'mean', options['normalization']),
              refeq_series(r_data, 'r', options['normalization'])]
    cut_risk = [[entry[2] for entry in group] for group in series]
    # Keep the R/Rinf=1 guides visible under automatic limits; manual limits win.
    limit_values = [values + ([[panel['reference_line']['value']]]
                    if options['normalization'] == 'Rinf' and panel['reference_line']['show'] else [])
                    for values, panel in zip(cut_risk, panels)]
    share_y = options['share_curve_y']
    share_bar = share_y and options['share_colorbar_y']
    if share_y:
        values = limit_values[0] + limit_values[1] + ([risk] if share_bar else [])
        shared = refeq_limits(values, options['shared_risk_ylim'])
        cut_limits = [shared, shared]
    else:
        cut_limits = [refeq_limits(values, panel['ylim'])
                      for values, panel in zip(limit_values, panels)]
    bar_limits = cut_limits[0] if share_bar else refeq_limits([risk], map_style['risk_ylim'])
    if options['color_limit_mode'] not in ('map', 'risk_ylim'):
        raise ValueError("color_limit_mode must be 'map' or 'risk_ylim'")
    clim = map_style['risk_clim']
    if clim is None and options['color_limit_mode'] == 'risk_ylim':
        clim = bar_limits
    # Colours span the map's own range by default, independently of risk-axis coordinates.
    color_limits = refeq_limits([risk], clim, padding=0.)
    norm = Normalize(*color_limits, clip=True)

    ratios = np.asarray(options['width_ratios'], dtype=float)
    gaps = np.asarray([options['map_bar_gap'], options['bar_curve_gap'], options['curve_gap']])
    if ratios.shape != (3,) or np.any(~np.isfinite(ratios)) or np.any(ratios <= 0):
        raise ValueError('width_ratios must contain three positive finite values')
    if np.any(~np.isfinite(gaps)) or np.any(gaps < 0):
        raise ValueError('Layout gaps must be finite and nonnegative')
    if not np.isfinite(options['bar_width']) or options['bar_width'] <= 0:
        raise ValueError('bar_width must be finite and positive')
    # Physical margins and gaps reserve space for the narrow colourbar and its label.
    width, height = float(options['width']), float(options['height'])
    margins = options['margins_inches']
    left, right, bottom, top = [float(margins[key]) for key in ('left', 'right', 'bottom', 'top')]
    if not np.all(np.isfinite([width, height, left, right, bottom, top])) or min(left, right, bottom, top) < 0:
        raise ValueError('Figure dimensions/margins must be finite; margins must be nonnegative')
    if min(width, height) <= 0:
        raise ValueError('Figure dimensions must be positive')
    has_status_legend = map_style['show_status_legend'] and any(
        np.any(np.isin(d['status'], ['infeasible', 'failed'])) for d in (data, mean_data, r_data))
    if has_status_legend:
        bottom += options['status_legend_space']
    # Optional restored cut ticks get their own room, independently of risk-axis sharing.
    gaps = gaps + [0., options['extra_y_tick_space']*options['show_middle_y_ticks'],
                        options['extra_y_tick_space']*options['show_right_y_ticks']]
    available_width = width-left-right-gaps.sum()
    axis_height = height-bottom-top
    if available_width <= 0 or axis_height <= 0:
        raise ValueError('Increase figure dimensions or reduce margins/gaps')
    unit = available_width/(ratios.sum()+options['bar_width'])
    panel_widths, bar_width = ratios*unit, options['bar_width']*unit
    bar_left = left+panel_widths[0]+gaps[0]
    middle_left = bar_left+bar_width+gaps[1]
    right_left = middle_left+panel_widths[1]+gaps[2]
    fig = plt.figure(figsize=(width, height), dpi=options['dpi'], layout='none')

    def add_axis(x, axis_width):
        return fig.add_axes([x/width, bottom/height, axis_width/width, axis_height/height])

    map_ax = add_axis(left, panel_widths[0])
    bar_ax = add_axis(bar_left, bar_width)
    mid_ax = add_axis(middle_left, panel_widths[1])
    right_ax = add_axis(right_left, panel_widths[2])
    for ax, group, panel, limits in zip((mid_ax, right_ax), series, panels, cut_limits):
        draw_refeq_cut(ax, group, options, panel)
        ax.set_ylim(limits)

    # Side strip uses the SAME colour normalization as the map while sharing y coordinates
    # with the cuts. Outside the map's range the palette saturates.
    draw_refeq_risk_strip(bar_ax, bar_limits, norm, options, map_style)
    labels = {'Rinf': r'Optimal risk $R_{\mathrm{opt}}/R_\infty$',
              'sigma2': r'Optimal risk $R_{\mathrm{opt}}/\sigma^2$',
              'none': r'Optimal risk $R_{\mathrm{opt}}$'}
    # The bar owns both the numerical ticks and their one risk label, inside its
    # reserved gap. Hidden ticks on panel b cannot pull the label over the strip.
    bar_ax.yaxis.set_label_position('right')
    bar_ax.set_ylabel(options['risk_label'] or labels[options['normalization']],
                      fontsize=options['label_size'], fontweight='bold',
                      labelpad=options['risk_labelpad'])
    ticks = options['shared_risk_ticks']
    if share_y and ticks is None: ticks = mid_ax.get_yticks()
    for ax, selected in ((mid_ax, ticks if share_y else panels[0]['yticks']),
                         (right_ax, ticks if share_y else panels[1]['yticks']),
                         (bar_ax, ticks if share_bar else map_style['risk_ticks'])):
        if selected is not None:
            selected = np.asarray(selected, dtype=float)
            if selected.ndim != 1 or np.any(~np.isfinite(selected)) or np.any(np.diff(selected) <= 0):
                raise ValueError('Risk ticks must be finite and increasing')
            ax.yaxis.set_major_locator(FixedLocator(selected))
        ax.yaxis.set_minor_locator(NullLocator())
    for ax, visible in ((mid_ax, options['show_middle_y_ticks']), (right_ax, options['show_right_y_ticks'])):
        ax.tick_params(axis='y', which='both', left=visible, labelleft=visible)

    x, y = data['mean_values'], data['r_values']
    z = np.ma.masked_invalid(risk)
    levels = map_style['contour_levels']
    if levels is None: levels = np.linspace(*color_limits, map_style['contour_count']+1)
    levels = np.asarray(levels, dtype=float)
    if levels.ndim != 1 or levels.size < 2 or np.any(~np.isfinite(levels)) or np.any(np.diff(levels) <= 0):
        raise ValueError('Contour levels must be finite and increasing')
    if map_style['fill'] == 'continuous':
        map_ax.pcolormesh(x, y, z, cmap=map_style['cmap'], norm=norm,
                          shading=map_style['shading'], rasterized=True)
    elif map_style['fill'] == 'bands':
        if np.any(np.isfinite(risk)):
            map_ax.contourf(x, y, z, levels=levels, cmap=map_style['cmap'], norm=norm,
                            corner_mask=False, extend='both')
    else:
        raise ValueError("Map fill must be 'continuous' or 'bands'")
    if map_style['show_contours'] and np.any(np.isfinite(risk)) and np.nanmin(risk) < np.nanmax(risk):
        map_ax.contour(x, y, z, levels=levels, corner_mask=False, **map_style['contour_style'])

    status_handles = []
    for status, colour, label in (('infeasible', map_style['infeasible_colour'], 'Infeasible'),
                                  ('failed', map_style['failed_colour'], 'Numerical failure')):
        mask = data['status'] == status
        if np.any(mask):
            map_ax.pcolormesh(x, y, np.ma.masked_where(~mask, np.ones(mask.shape)),
                cmap=ListedColormap([colour]), vmin=0., vmax=1., shading='nearest', zorder=3)
        if np.any(mask) or any(np.any(d['status'] == status) for d in (mean_data, r_data)):
            status_handles.append(Patch(facecolor=colour, label=label))
    known = data['preparation_status'] != 'failed'
    if (map_style['show_feasibility_boundary'] and np.any(data['feasible'] & known)
            and np.any(~data['feasible'] & known)):
        map_ax.contour(x, y, np.ma.masked_where(~known, data['feasible'].astype(float)),
                       levels=[.5], corner_mask=False, zorder=4, **map_style['boundary_style'])
    if map_style['show_cut_guides']:
        for group, panel, direction in zip(series, panels, ('h', 'v')):
            for (value, *_), colour in zip(group, refeq_colours(len(group), panel)):
                guide = dict(color=panel['curve_overrides'].get(value, {}).get('color', colour))
                guide.update(map_style['cut_guide_style'])
                line = getattr(map_ax, 'ax'+direction+'line')(value, **guide)
                halo = dict(map_style['cut_guide_outline'])
                if halo.pop('show'):
                    line.set_path_effects([path_effects.Stroke(**halo), path_effects.Normal()])
    map_ax.set_yscale('log')
    map_ax.set_xlim(map_style['xlim'] or (x[0], x[-1]))
    map_ax.set_ylim(map_style['ylim'] or (y[0], y[-1]))
    map_ax.set_xlabel(map_style['xlabel'], fontsize=options['label_size'], fontweight='bold')
    map_ax.set_ylabel(map_style['ylabel'], fontsize=options['label_size'], fontweight='bold')
    for axis in ('x', 'y'):
        if map_style[axis+'ticks'] is not None:
            getattr(map_ax, 'set_'+axis+'ticks')(map_style[axis+'ticks'])
    annotation = dict(map_style['annotation'])
    if annotation.pop('show'):
        position = annotation.pop('xy')
        text = annotation.pop('text_format').format(**data['metadata']['base'])
        map_ax.text(*position, text, transform=map_ax.transAxes, zorder=10, **annotation)
    if status_handles and map_style['show_status_legend']:
        fig.legend(handles=status_handles, fontsize=options['legend_size'], **map_style['status_legend'])
    for ax in (map_ax, bar_ax, mid_ax, right_ax):
        ax.tick_params(labelsize=options['tick_size'])
        ax.grid(False, which='both')
    for ax, label in zip((map_ax, mid_ax, right_ax), options['panel_labels']):
        ax.text(*options['panel_label_xy'], label, transform=ax.transAxes,
                fontsize=options['panel_label_size'], fontweight='bold')
    return fig


## Presentation controls

In [ ]:
# Presentation changes use the same numerical arrays.
# share_curve_y/share_colorbar_y control risk axes independently of map colours.
REF_EQ_LAYOUT = {'width': 10.0,
 'height': 3.3,
 'dpi': 140,
 'width_ratios': [1.0, 1.2, 1.2],
 'bar_width': 0.055,
 'bar_resolution': 512,
 'margins_inches': {'left': 0.65, 'right': 0.1, 'bottom': 0.7, 'top': 0.25},
 'map_bar_gap': 0.07,
 'bar_curve_gap': 0.7,
 'curve_gap': 0.1,
 'extra_y_tick_space': 0.3,
 'status_legend_space': 0.3,
 'colorbar': {'extend': 'both',
              'extension_size': 0.025,
              'edgecolor': '0.25',
              'linewidth': 0.7,
              'tickpad': 3.0,
              'tick_length': 3.5},
 'normalization': 'Rinf',
 'risk_label': None,
 'risk_labelpad': 6.0,
 'share_curve_y': True,
 'share_colorbar_y': True,
 'color_limit_mode': 'map',
 'shared_risk_ylim': None,
 'shared_risk_ticks': None,
 'show_middle_y_ticks': False,
 'show_right_y_ticks': False,
 'label_size': 11,
 'tick_size': 11,
 'legend_size': 11,
 'title_size': 12,
 'panel_labels': ['a', 'b', 'c'],
 'panel_label_xy': [-0.02, 1.03],
 'panel_label_size': 12}

REF_EQ_MAP_STYLE = {'cmap': 'viridis',
 'fill': 'continuous',
 'shading': 'gouraud',
 'risk_clim': None,
 'risk_ylim': None,
 'risk_ticks': None,
 'show_contours': True,
 'contour_count': 20,
 'contour_levels': None,
 'contour_style': {'colors': '0.2', 'linewidths': 0.5, 'alpha': 0.5},
 'show_feasibility_boundary': True,
 'boundary_style': {'colors': 'black', 'linewidths': 0.8},
 'infeasible_colour': '#dedede',
 'failed_colour': '#d8758b',
 'show_cut_guides': True,
 'cut_guide_style': {'linestyle': '--',
                     'linewidth': 1.0,
                     'alpha': 1.0,
                     'zorder': 6,
                     'clip_on': False},
 'cut_guide_outline': {'show': True, 'linewidth': 2.0, 'foreground': 'white', 'alpha': 0.85},
 'xlabel': 'Differing-site mean contribution\n$u=m_{\\neq}=\\tilde m_{\\neq}$',
 'ylabel': 'Aspect ratio $r=qL/n$',
 'xlim': None,
 'ylim': None,
 'xticks': [0.0, 0.2, 0.4, 0.6],
 'yticks': [0.1, 1.0, 10.0],
 'annotation': {'show': False,
                'text_format': '$m=0,\\ F=\\tilde F={F:g}$\n'
                               '$F_{{\\neq}}=\\tilde F_{{\\neq}}={Fneq:g}$',
                'xy': [0.04, 0.96],
                'fontsize': 10,
                'ha': 'left',
                'va': 'top',
                'bbox': {'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85, 'pad': 2}},
 'show_status_legend': True,
 'status_legend': {'loc': 'best', 'bbox_to_anchor': [0.0, 0.0], 'ncol': 2, 'frameon': False}}

REF_EQ_MEAN_PANEL = {'xlabel': '$u$',
 'xscale': 'linear',
 'xlim': None,
 'ylim': None,
 'xticks': [0.0, 0.2, 0.4, 0.6],
 'yticks': None,
 'title': '',
 'cmap': 'viridis',
 'colour_range': [0.12, 0.88],
 'colours': None,
 'linewidth': 1.8,
 'linestyle': '-',
 'label_format': '$r={value:g}$',
 'curve_overrides': {},
 'show_legend': True,
 'legend': {'loc': [0.62, 0.13], 'frameon': True},
 'show_isolated_points': True,
 'isolated_marker': 'o',
 'isolated_marker_size': 4.0,
 'reference_line': {'show': True,
                    'value': 1.0,
                    'color': '0.6',
                    'linestyle': ':',
                    'linewidth': 1.0,
                    'zorder': 0}}

REF_EQ_R_PANEL = {'xlabel': '$r$',
 'xscale': 'log',
 'xlim': None,
 'ylim': None,
 'xticks': [0.1, 1.0, 10.0],
 'yticks': None,
 'title': '',
 'cmap': 'cividis',
 'colour_range': [0.15, 0.85],
 'colours': None,
 'linewidth': 1.8,
 'linestyle': '-',
 'label_format': '$u={value:g}$',
 'curve_overrides': {},
 'show_legend': True,
 'legend': {'loc': 'best', 'frameon': True},
 'show_isolated_points': True,
 'isolated_marker': 'o',
 'isolated_marker_size': 4.0,
 'reference_line': {'show': True,
                    'value': 1.0,
                    'color': '0.6',
                    'linestyle': ':',
                    'linewidth': 1.0,
                    'zorder': 0}}


## Draw from the loaded arrays

In [ ]:
reference_figure = draw_reference_equal_r(
    MAP_DATA, MEAN_DATA, R_DATA, REF_EQ_LAYOUT, REF_EQ_MAP_STYLE,
    [REF_EQ_MEAN_PANEL, REF_EQ_R_PANEL])
FIGURES['reference_equal_mean_r'] = reference_figure
display(reference_figure)
plt.close(reference_figure)


## Save the displayed PDF

In [ ]:
PDF_FIGURES = {'reference_equal_mean_r': 'reference_effects_new.pdf'}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for key, filename in PDF_FIGURES.items():
    if key not in FIGURES:
        raise RuntimeError(f'Display {key} before saving.')
    FIGURES[key].savefig(OUTPUT_DIR / filename, format='pdf', bbox_inches='tight')
    print(Path('output') / filename)


## Optional numerical export

In [ ]:
# Optional: save recomputed data without overwriting the supplied archive.
SAVE_RECOMPUTED_DATA = False
if SAVE_RECOMPUTED_DATA:
    save_results(OUTPUT_DIR / 'reference_effects_arrays',
        dict(map=MAP_DATA, mean_cuts=MEAN_DATA, r_cuts=R_DATA),
        metadata=dict(settings=reference_settings(),
            source_identities=dict(map=MAP_ID, mean_cuts=MEAN_ID, r_cuts=R_ID),
            figure='reference_effects_new.pdf'))
